In [ ]:
!pip install mediapipe==0.10.14

In [ ]:
!pip install opencv-python

In [ ]:
import cv2
import mediapipe as mp
import math
import numpy as np

# Initialize MediaPipe Pose
mp_pose = mp.solutions.pose
pose = mp_pose.Pose(min_detection_confidence=0.5, min_tracking_confidence=0.5)

def calculate_angle(a, b, c):
    """Calculates the angle between three points."""
    a = np.array(a) # First point (e.g., Shoulder)
    b = np.array(b) # Mid point / Vertex (e.g., Elbow)
    c = np.array(c) # End point (e.g., Wrist)

    radians = math.atan2(c[1] - b[1], c[0] - b[0]) - math.atan2(a[1] - b[1], a[0] - b[0])
    angle = np.abs(radians * 180.0 / math.pi)

    if angle > 180.0:
        angle = 360 - angle

    return round(angle, 2)

def process_video(video_path, point1, point2, point3):
    """Processes a video and returns the max and min angles of a specific joint."""
    cap = cv2.VideoCapture(video_path)
    angles = []

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        # Convert the BGR image to RGB
        image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        image.flags.writeable = False

        # Process the image and detect the pose
        results = pose.process(image)

        # Extract the landmarks
        if results.pose_landmarks:
            landmarks = results.pose_landmarks.landmark

            try:
                # Grab the coordinates for the three required joints
                a = [landmarks[point1].x, landmarks[point1].y]
                b = [landmarks[point2].x, landmarks[point2].y]
                c = [landmarks[point3].x, landmarks[point3].y]

                # Calculate and store the angle for this frame
                angle = calculate_angle(a, b, c)
                angles.append(angle)
            except Exception as e:
                pass

    cap.release()

    if not angles:
        return None, None

    return max(angles), min(angles)

# --- HOW TO USE IT ---
# Example: Upload a test video named 'test_curl.mp4' to your Colab directory.
# We will track the Left Arm: Shoulder (11), Elbow (13), Wrist (15)

video_file = '/content/123.mp4' # Replace with your actual video path

try:
    max_angle, min_angle = process_video(video_file, 23, 11, 13)
    print(f"Video Processing Complete!")
    print(f"Maximum Angle Achieved: {max_angle} degrees")
    print(f"Minimum Angle Achieved: {min_angle} degrees")

    # This is the exact data you will inject into your LLM prompt later!
except Exception as e:
    print("Please make sure you have a valid video file path.")

ImportError: cannot import name 'runtime_version' from 'google.protobuf' (/usr/local/lib/python3.12/dist-packages/google/protobuf/__init__.py)

In [ ]:
!pip install edge-tts

In [ ]:
# The raw output from your Qwen model
raw_feedback = """Score: 65/100

Angle Analysis:
Detected angle: 104.93°
Target range: 0-90°
Result: above safe range

Biomechanical Interpretation:
This indicates the movement range is not aligned with the target mechanics.

Coaching Feedback:
There is a form issue here. The detected angle was 104.93° which is outside the safe range of 0-90°. This deviation may increase stress on the joint. Detected issue: lifting above safe range. Correction: Stay below pain level. Try to maintain consistent technique across repetitions."""

# Extract only the text after "Coaching Feedback:"
# We use try/except to ensure it doesn't break if the model formats the output slightly differently
try:
    spoken_feedback = raw_feedback.split("Coaching Feedback:\n")[1].strip()
except IndexError:
    spoken_feedback = raw_feedback # Fallback to reading the whole thing

print("Text to speak:", spoken_feedback)

Text to speak: There is a form issue here. The detected angle was 104.93° which is outside the safe range of 0-90°. This deviation may increase stress on the joint. Detected issue: lifting above safe range. Correction: Stay below pain level. Try to maintain consistent technique across repetitions.


In [ ]:
import edge_tts
import asyncio
from IPython.display import Audio, display

async def create_coach_audio(text, output_filename="coach_audio.mp3"):
    # "en-US-ChristopherNeural" is a solid, clear male voice.
    # You can change this to a female voice like "en-US-AriaNeural"
    communicate = edge_tts.Communicate(text, "en-US-ChristopherNeural")

    # Save the audio file
    await communicate.save(output_filename)

    # Display an audio player right inside Colab
    print("Playing feedback...")
    display(Audio(output_filename, autoplay=True))

# Run the async function with your extracted text
await create_coach_audio(spoken_feedback)

Playing feedback...


In [ ]:
# 1. Mount your Google Drive
from google.colab import drive
drive.mount('/content/drive')

# # 2. Install the SAFE, STABLE release directly from PyPI
! pip install --upgrade --no-cache-dir unsloth unsloth-zoo
! pip install --no-deps xformers trl peft accelerate bitsandbytes datasets


Mounted at /content/drive
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.0/56.0 kB 21.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.4/67.4 MB 389.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 428.0/428.0 kB 414.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 262.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 441.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 398.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 273.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 262.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 273.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 276.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.0/225.0 kB 224.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 241.5 MB/s eta 0:00:00
   ━━━━━━━━━━━

In [ ]:
import shutil
import os

In [ ]:
from unsloth import FastLanguageModel

# Point this to your exact Drive folder
drive_model_path = "/content/drive/MyDrive/graduation_project/rehab_coach_lora"

print("Loading model from Drive...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = drive_model_path,
    max_seq_length = 2048,
    load_in_4bit = True,
)

FastLanguageModel.for_inference(model)
print("Model successfully loaded and ready for testing!")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Loading model from Drive...
==((====))==  Unsloth 2026.5.2: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.55G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/271 [00:00<?, ?B/s]

Unsloth 2026.5.2 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


Model successfully loaded and ready for testing!


In [ ]:
# Save the fine-tuned adapters to a folder
model.save_pretrained("rehab_coach_lora")
tokenizer.save_pretrained("rehab_coach_lora")

print("Model adapters saved successfully to the 'rehab_coach_lora' folder!")

Model adapters saved successfully to the 'rehab_coach_lora' folder!


In [ ]:
import os

drive_save_path = "/content/drive/MyDrive/graduation_project/rehab_coach_lora"

# Create the directory if it doesn't exist
os.makedirs(drive_save_path, exist_ok=True)

# Save the fine-tuned adapters to Google Drive
model.save_pretrained(drive_save_path)
tokenizer.save_pretrained(drive_save_path)

print(f"Model adapters saved successfully to '{drive_save_path}' in Google Drive!")

Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/graduation_project/rehab_coach_lora/tokenizer_config.json.


Model adapters saved successfully to '/content/drive/MyDrive/graduation_project/rehab_coach_lora' in Google Drive!


In [ ]:
!pip install --upgrade --no-cache-dir "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers trl peft accelerate bitsandbytes

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-4cis7n4v/unsloth_f5634949159343e2bbc2e13a5a9c388e
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-4cis7n4v/unsloth_f5634949159343e2bbc2e13a5a9c388e
  Resolved https://github.com/unslothai/unsloth.git to commit 739ebeea8241d125c1bb8fa401207e62dcacb326
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# الكود اللي حنعتمده
import cv2
import mediapipe as mp
import numpy as np
import pandas as pd
import math
from google.colab.patches import cv2_imshow
import gc
import edge_tts
import asyncio
from IPython.display import Audio, display, Video, HTML
import time
from moviepy.editor import VideoFileClip, AudioFileClip, concatenate_audioclips
import os
import torch
from unsloth import FastLanguageModel

# # تنظيف الذاكرة قبل البدء
torch.cuda.empty_cache()

# حذف النماذج السابقة إذا وجدت
if 'model' in locals() and model is not None:
    del model
if 'tokenizer' in locals() and tokenizer is not None:
    del tokenizer
gc.collect()
torch.cuda.empty_cache()

# تحميل النموذج
model_path = "/content/drive/MyDrive/graduation_project/rehab_coach_lora_FINAL"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_path,
    max_seq_length = 1024,
    load_in_4bit = True,
    device_map = "cuda:0",
    low_cpu_mem_usage = True,
)

FastLanguageModel.for_inference(model)

# -------------------------------------------------------------------
# خريطة المفاصل
# -------------------------------------------------------------------
JOINT_MAP = {
    "shoulder_angle_left": (13, 11, 23),
    "shoulder_angle_right": (14, 12, 24),
    "shoulder_flexion_left": (13, 11, 12),
    "shoulder_flexion_right": (14, 12, 11),

    "elbow_left": (11, 13, 15),
    "elbow_right": (12, 14, 16),

    "knee_left": (23, 25, 27),
    "knee_right": (24, 26, 28),

    "hip_left": (11, 23, 25),
    "hip_right": (12, 24, 26),
    "hip_abduction_left": (23, 24, 25),
    "hip_abduction_right": (24, 23, 26),
}

# -------------------------------------------------------------------
# قاموس التمارين
# -------------------------------------------------------------------
exercise_rules = {
    "shoulder_based": {
        "arm_abduction": {
            "joints": ["shoulder_angle_left", "shoulder_angle_right"],
            "up_threshold": 80,
            "down_threshold": 20,
            "min_angle": 15,
            "max_angle": 95,
            "type": "upper",
            "description": "Arm abduction (lifting arms sideways)"
        },
        "shoulder_flexion": {
            "joints": ["shoulder_flexion_left", "shoulder_flexion_right"],
            "up_threshold": 90,
            "down_threshold": 10,
            "min_angle": 0,
            "max_angle": 180,
            "type": "upper",
            "description": "Shoulder flexion (lifting arms forward)"
        },
        "arm_vw": {
            "joints": ["shoulder_angle_left", "shoulder_angle_right"],
            "up_threshold": 120,
            "down_threshold": 60,
            "min_angle": 45,
            "max_angle": 135,
            "type": "upper",
            "description": "V-W arm exercise"
        },
        "arm_circles": {
            "joints": ["shoulder_angle_left", "shoulder_angle_right"],
            "up_threshold": 60,
            "down_threshold": 120,
            "min_angle": 30,
            "max_angle": 150,
            "type": "upper",
            "description": "Arm circles"
        },
        "arm_half_circles": {
            "joints": ["shoulder_angle_left", "shoulder_angle_right"],
            "up_threshold": 60,
            "down_threshold": 100,
            "min_angle": 30,
            "max_angle": 120,
            "type": "upper",
            "description": "Half arm circles"
        }
    },

    "elbow_based": {
        "pushups": {
            "joints": ["elbow_left", "elbow_right"],
            "up_threshold": 160,
            "down_threshold": 80,
            "min_angle": 70,
            "max_angle": 170,
            "type": "upper",
            "description": "Push-ups"
        },
        "bench_press": {
            "joints": ["elbow_left", "elbow_right"],
            "up_threshold": 150,
            "down_threshold": 80,
            "min_angle": 70,
            "max_angle": 160,
            "type": "upper",
            "description": "Bench press"
        },
        "shoulder_press": {
            "joints": ["elbow_left", "elbow_right"],
            "up_threshold": 160,
            "down_threshold": 90,
            "min_angle": 80,
            "max_angle": 170,
            "type": "upper",
            "description": "Shoulder press"
        },
        "bicep_curl": {
            "joints": ["elbow_left", "elbow_right"],
            "up_threshold": 40,
            "down_threshold": 150,
            "min_angle": 30,
            "max_angle": 160,
            "type": "upper",
            "description": "Bicep curl"
        },
        "triceps_pushdown": {
            "joints": ["elbow_left", "elbow_right"],
            "up_threshold": 160,
            "down_threshold": 40,
            "min_angle": 30,
            "max_angle": 170,
            "type": "upper",
            "description": "Triceps pushdown"
        },
        "lat_pulldown": {
            "joints": ["elbow_left", "elbow_right"],
            "up_threshold": 150,
            "down_threshold": 60,
            "min_angle": 45,
            "max_angle": 160,
            "type": "upper",
            "description": "Lat pulldown"
        }
    },

    "knee_based": {
        "bodyweight_squats": {
            "joints": ["knee_left", "knee_right"],
            "up_threshold": 120,
            "down_threshold": 60,
            "min_angle": 40,
            "max_angle": 130,
            "type": "lower",
            "description": "Bodyweight squats"
        },
        "squats": {
            "joints": ["knee_left", "knee_right"],
            "up_threshold": 160,
            "down_threshold": 80,
            "min_angle": 70,
            "max_angle": 170,
            "type": "lower",
            "description": "Squats"
        },
        "leg_lunge": {
            "joints": ["knee_left", "knee_right"],
            "up_threshold": 165,
            "down_threshold": 95,
            "min_angle": 85,
            "max_angle": 175,
            "type": "lower",
            "description": "Forward lunge"
        },
        "lunge": {
            "joints": ["knee_left"],
            "up_threshold": 140,
            "down_threshold": 85,
            "min_angle": 70,
            "max_angle": 175,
            "type": "lower",
            "description": "Lunge exercise"
        },
        "butt_kicks": {
            "joints": ["knee_left", "knee_right"],
            "up_threshold": 60,
            "down_threshold": 150,
            "min_angle": 45,
            "max_angle": 160,
            "type": "lower",
            "description": "Butt kicks"
        },
        "high_knee": {
            "joints": ["knee_left", "knee_right"],
            "up_threshold": 80,
            "down_threshold": 160,
            "min_angle": 70,
            "max_angle": 170,
            "type": "lower",
            "description": "High knees"
        },
        "leg_extension": {
            "joints": ["knee_left", "knee_right"],
            "up_threshold": 160,
            "down_threshold": 100,
            "min_angle": 90,
            "max_angle": 175,
            "type": "lower",
            "description": "Leg extension"
        }
    },

    "hip_based": {
        "leg_swing": {
            "joints": ["hip_left", "hip_right"],
            "up_threshold": 80,
            "down_threshold": 10,
            "min_angle": 0,
            "max_angle": 90,
            "type": "lower",
            "description": "Leg swing"
        },
        "jumping_jacks": {
            "joints": ["hip_abduction_left", "hip_abduction_right"],
            "up_threshold": 140,
            "down_threshold": 30,
            "min_angle": 20,
            "max_angle": 150,
            "type": "lower",
            "description": "Jumping jacks"
        },
        "leg_abduction": {
            "joints": ["hip_abduction_left", "hip_abduction_right"],
            "up_threshold": 30,
            "down_threshold": 5,
            "min_angle": 0,
            "max_angle": 45,
            "type": "lower",
            "description": "Leg abduction"
        }
    }
}

# -------------------------------------------------------------------
# -------------------------------------------------------------------
def calculate_angle(a, b, c):
    a = np.array([a[0], a[1]])
    b = np.array([b[0], b[1]])
    c = np.array([c[0], c[1]])

    ba = a - b
    bc = c - b

    norm_ba = np.linalg.norm(ba)
    norm_bc = np.linalg.norm(bc)

    if norm_ba < 0.01 or norm_bc < 0.01:
        return 0

    cosine = np.dot(ba, bc) / (norm_ba * norm_bc)
    cosine = np.clip(cosine, -1.0, 1.0)
    angle = np.arccos(cosine)

    return int(np.degrees(angle))

def calculate_hip_abduction(hip_left, hip_right, knee):
    hip_left = np.array([hip_left[0], hip_left[1]])
    hip_right = np.array([hip_right[0], hip_right[1]])
    knee = np.array([knee[0], knee[1]])

    hip_vector = hip_right - hip_left

    if np.linalg.norm(hip_left - knee) < np.linalg.norm(hip_right - knee):
        knee_vector = knee - hip_left
    else:
        knee_vector = knee - hip_right

    dot = np.dot(hip_vector, knee_vector)
    norm_product = np.linalg.norm(hip_vector) * np.linalg.norm(knee_vector)

    if norm_product < 0.01:
        return 0

    cos_angle = abs(dot) / norm_product
    cos_angle = np.clip(cos_angle, -1.0, 1.0)
    angle = np.arccos(cos_angle)

    return int(np.degrees(angle))

def smooth_angle(history, new_val, window=3):
    history.append(new_val)
    if len(history) > window:
        history.pop(0)
    return np.mean(history)

def detect_peaks_valleys(angles, window=5):

    if len(angles) < window:
        return [], []

    peaks = []
    valleys = []

    for i in range(window, len(angles) - window):
        # قمة
        if angles[i] > max(angles[i-window:i] + angles[i+1:i+window+1]):
            peaks.append((i, angles[i]))
        # قاع
        if angles[i] < min(angles[i-window:i] + angles[i+1:i+window+1]):
            valleys.append((i, angles[i]))

    return peaks, valleys

def count_reps_adaptive(angle_history, exercise_name, stage, last_rep_frame, frame, min_frames, is_lower_body=True):

    if len(angle_history) < 10:
        return stage, False, last_rep_frame

    # حساب العتبات ديناميكياً من البيانات
    recent = angle_history[-20:]
    max_angle = max(recent)
    min_angle = min(recent)
    range_motion = max_angle - min_angle  # إضافة تعريف range_motion
    if range_motion < 35:
      return stage, False, last_rep_frame

    # عتبات تكيفية
    up_th = max_angle - (range_motion * 0.15)  # 75% من أقصى ارتفاع (تعديل)
    down_th = min_angle + (range_motion * 0.15)  # 25% من أدنى انخفاض (تعديل)

    current = np.mean(angle_history[-3:])

    new_stage = stage
    rep_counted = False

    # تحسين لمنطق العد لتمارين الرجلين
    if exercise_name in ["lunge", "leg_lunge", "squats", "bodyweight_squats"]:
        # لتمارين الرجلين: النزول (ثني الركبة) ثم الصعود (فرد الركبة)
        if current < down_th:  # الركبة مثنية (وضعية النزول)
            new_stage = "down"
        elif current > up_th and stage == "down":  # الركبة مفرودة (وضعية الصعود) بعد النزول
            new_stage = "up"
            if frame - last_rep_frame > min_frames:
                rep_counted = True
                last_rep_frame = frame
                print(f"  ↳ تكرار عند: max={max_angle:.0f}°, min={min_angle:.0f}°, up_th={up_th:.0f}°, down_th={down_th:.0f}°")
    else:
        # للتمارين الأخرى
        if current > up_th:
            new_stage = "up"
        elif current < down_th and stage == "up":
            new_stage = "down"
            if frame - last_rep_frame > min_frames:
                rep_counted = True
                last_rep_frame = frame
                print(f"  ↳ تكرار عند: max={max_angle:.0f}°, min={min_angle:.0f}°, up_th={up_th:.0f}°, down_th={down_th:.0f}°")

    return new_stage, rep_counted, last_rep_frame

def draw_info(image, name, angle, reps, stage, min_a, max_a, angle_type, debug_info="", angle_history=None):
    h, w = image.shape[:2]

    overlay = image.copy()
    cv2.rectangle(overlay, (10, 10), (450, 190), (0, 0, 0), -1)
    cv2.addWeighted(overlay, 0.6, image, 0.4, 0, image)

    cv2.rectangle(image, (10, 10), (450, 190), (255, 255, 255), 1)

    title = name.replace('_', ' ').title()
    cv2.putText(image, f"{title}", (20, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 1)
    cv2.putText(image, f"{angle_type}", (280, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 255), 1)
    cv2.putText(image, f"REPS: {reps}", (20, 55), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
    cv2.putText(image, f"Angle: {int(angle)}°", (20, 80), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 0), 1)

    color = (0, 255, 0) if min_a <= angle <= max_a else (0, 0, 255)
    cv2.putText(image, f"Safe: {min_a}-{max_a}", (20, 105), cv2.FONT_HERSHEY_SIMPLEX, 0.4, color, 1)

    if stage:
        stage_color = (0, 255, 255) if stage == "up" else (255, 165, 0)
        cv2.putText(image, f"STAGE: {stage.upper()}", (280, 80), cv2.FONT_HERSHEY_SIMPLEX, 0.5, stage_color, 1)

    # إضافة معلومات التشخيص
    if debug_info:
        cv2.putText(image, debug_info, (20, 130), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (200, 200, 200), 1)

    if angle_history and len(angle_history) > 10:
        recent_min = min(angle_history[-20:])
        recent_max = max(angle_history[-20:])
        cv2.putText(image, f"Range: {recent_min:.0f}-{recent_max:.0f}", (20, 150), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (200, 200, 200), 1)
        cv2.putText(image, f"Thresholds: {recent_min + (recent_max-recent_min)*0.25:.0f}-{recent_max - (recent_max-recent_min)*0.25:.0f}", (20, 170), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (200, 200, 200), 1)

    return image

def add_angle(image, pos, angle, is_main_joint=False):
    x, y = int(pos[0]), int(pos[1])
    # خلفية مختلفة للمفصل الرئيسي
    if is_main_joint:
        cv2.rectangle(image, (x-25, y-25), (x+35, y+10), (0, 0, 0), -1)
        cv2.rectangle(image, (x-25, y-25), (x+35, y+10), (0, 255, 0), 2)
        cv2.putText(image, f"{int(angle)}°", (x-20, y-10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)
    else:
        cv2.rectangle(image, (x-20, y-20), (x+30, y+5), (0, 0, 0), -1)
        cv2.rectangle(image, (x-20, y-20), (x+30, y+5), (0, 255, 255), 1)
        cv2.putText(image, f"{int(angle)}°", (x-15, y-5), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (0, 255, 255), 1)

def generate_ai_feedback(exercise_name, category, target_min, target_max, min_achieved, max_achieved, total_reps):
    """توليد تقييم باستخدام الذكاء الاصطناعي"""

    # تحديد نوع التمرين
    exercise_type = ""
    for cat, exercises in exercise_rules.items():
        if exercise_name in exercises:
            exercise_type = exercises[exercise_name].get('type', 'unknown')
            break

    test_instruction = (f"Evaluate the trainee's performance for the {category} exercise: {exercise_name}. "
                        f"This is a {exercise_type} body exercise. "
                        f"The target safe range is {target_min} to {target_max} degrees. "
                        f"The trainee achieved a range of {min_achieved:.1f} to {max_achieved:.1f} degrees. "
                        f"Total repetitions completed: {total_reps}. "
                        f"Provide specific feedback on form and suggestions for improvement. "
                        f"Be encouraging and constructive.")

    prompt = f"<|im_start|>user\n{test_instruction}<|im_end|>\n<|im_start|>assistant\n"
    inputs = tokenizer([prompt], return_tensors="pt").to("cuda")

    outputs = model.generate(**inputs, max_new_tokens=200, use_cache=True, temperature=0.7)
    response = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]
    return response.split("assistant\n")[-1].strip()

async def create_coach_audio(text, output_filename="coach_audio.mp3"):
    """تحويل النص إلى كلام"""
    communicate = edge_tts.Communicate(text, "en-US-ChristopherNeural")
    await communicate.save(output_filename)
    print(f"🔊 تم حفظ الملف الصوتي: {output_filename}")
    return output_filename

def extract_feedback_for_tts(report_text):
    """استخراج الجزء المناسب للصوت"""
    sentences = report_text.split('. ')
    spoken_part = '. '.join(sentences[:3])
    if len(spoken_part) > 300:
        spoken_part = spoken_part[:300] + '...'
    return spoken_part

def add_audio_to_video(video_path, audio_path, output_path):
    """دمج الصوت مع الفيديو"""
    try:
        print("🎵 جاري دمج الصوت مع الفيديو...")

        if not os.path.exists(video_path) or not os.path.exists(audio_path):
            return None

        video = VideoFileClip(video_path)
        audio = AudioFileClip(audio_path)

        if audio.duration < video.duration:
            repeats = int(video.duration / audio.duration) + 1
            audio_clips = [audio] * repeats
            audio = concatenate_audioclips(audio_clips)
            audio = audio.subclip(0, video.duration)

        audio = audio.volumex(1.5)
        final_video = video.set_audio(audio)

        final_video.write_videofile(
            output_path,
            codec='libx264',
            audio_codec='aac',
            verbose=False,
            logger=None
        )

        video.close()
        audio.close()
        final_video.close()

        return output_path

    except Exception as e:
        print(f"❌ خطأ في دمج الصوت: {e}")
        return None

def find_exercise(exercise_name):
    for category, exercises in exercise_rules.items():
        if exercise_name in exercises:
            return category, exercises[exercise_name]
    return None, None

# -------------------------------------------------------------------
# الدالة الرئيسية مع التشخيص المحسن
# -------------------------------------------------------------------
def analyze_with_ai_feedback(video_path, exercise_name):
    """تحليل التمرين مع تقييم الذكاء الاصطناعي وتشخيص المشاكل"""

    # البحث عن التمرين
    category, ex_info = find_exercise(exercise_name)
    if category is None:
        print(f"❌ التمرين '{exercise_name}' غير موجود")
        return

    print(f"\n{'='*60}")
    print(f"🏋️  تحليل تمرين: {exercise_name}")
    print(f"📌 {ex_info['description']}")
    print(f"{'='*60}\n")

    # إعداد MediaPipe
    mp_pose = mp.solutions.pose
    pose = mp_pose.Pose(min_detection_confidence=0.5, min_tracking_confidence=0.5)

    # فتح الفيديو
    cap = cv2.VideoCapture(video_path)
    fps = int(cap.get(cv2.CAP_PROP_FPS))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    # إعداد الفيديو الناتج
    temp_video = f"/content/{exercise_name}_temp.mp4"
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(temp_video, fourcc, fps, (width, height))

    # متغيرات العد - تحسين لتمارين الرجلين
    reps = 0
    stage = None  # المرحلة: down (نزول) أو up (صعود)
    last_rep_frame = -fps
    all_angles = []  # تاريخ الزوايا للركبة الرئيسية

    # تخزين زوايا كل ركبة على حدة للتشخيص
    left_knee_angles = []
    right_knee_angles = []

    joint_histories = {j: [] for j in ex_info['joints']}
    frame_count = 0

    # للتشخيص
    angle_samples = []

    # تحديد ما إذا كان التمرين للرجلين
    is_lower_body = ex_info.get('type') == 'lower'

    print(f"📊 إجمالي الإطارات: {total_frames}")
    print(f"📐 عتبات التمرين الأصلية: النزول < {ex_info['down_threshold']}°, الصعود > {ex_info['up_threshold']}°")
    print("\n🔄 جاري تحليل الحركة...")

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = pose.process(rgb)

        if results.pose_landmarks:
            landmarks = results.pose_landmarks.landmark
            frame_angles = []

            # معالجة خاصة لـ leg_abduction
            if exercise_name == "leg_abduction":
                hip_l = [landmarks[23].x, landmarks[23].y]
                hip_r = [landmarks[24].x, landmarks[24].y]
                knee_l = [landmarks[25].x, landmarks[25].y]
                knee_r = [landmarks[26].x, landmarks[26].y]

                angle_l = calculate_hip_abduction(hip_l, hip_r, knee_l)
                angle_r = calculate_hip_abduction(hip_l, hip_r, knee_r)
                main_angle = max(angle_l, angle_r)

                pos_l = (int(knee_l[0]*width), int(knee_l[1]*height))
                pos_r = (int(knee_r[0]*width), int(knee_r[1]*height))
                add_angle(frame, pos_l, angle_l)
                add_angle(frame, pos_r, angle_r)

            else:
                # تخزين زوايا كل ركبة على حدة
                left_angle = None
                right_angle = None

                for joint in ex_info['joints']:
                    if joint in JOINT_MAP:
                        p1, p2, p3 = JOINT_MAP[joint]
                        a = [landmarks[p1].x, landmarks[p1].y]
                        b = [landmarks[p2].x, landmarks[p2].y]
                        c = [landmarks[p3].x, landmarks[p3].y]

                        angle = calculate_angle(a, b, c)

                        if joint in joint_histories:
                            angle = smooth_angle(joint_histories[joint], angle)

                        frame_angles.append(angle)
                        pos = (int(b[0]*width), int(b[1]*height))

                        # تخزين زوايا الركبتين
                        if joint == "knee_left":
                            left_angle = angle
                            left_knee_angles.append(angle)
                        elif joint == "knee_right":
                            right_angle = angle
                            right_knee_angles.append(angle)

                        # تحديد ما إذا كان هذا المفصل هو المفصل الرئيسي للعد
                        is_main = False
                        if exercise_name in ["lunge", "leg_lunge", "squats", "bodyweight_squats"]:
                            # لتمارين الرجلين، نستخدم الركبة الأكثر انثناءً للعد
                            if left_angle is not None and right_angle is not None:
                                is_main = (joint == "knee_left" and left_angle < right_angle) or \
                                         (joint == "knee_right" and right_angle < left_angle)

                        add_angle(frame, pos, angle, is_main)

                # للتمارين التي تعتمد على الركبتين، نستخدم الركبة الأكثر انثناءً للعد
                if exercise_name in ["lunge", "leg_lunge", "squats", "bodyweight_squats"]:
                    if left_angle is not None and right_angle is not None:
                        # نستخدم الزاوية الأصغر (الأكثر انثناءً) للعد
                        main_angle = min(left_angle, right_angle)
                    else:
                        main_angle = np.mean(frame_angles) if frame_angles else 0
                else:
                    main_angle = np.mean(frame_angles) if frame_angles else 0

            if main_angle > 0:
                all_angles.append(main_angle)

                # جمع عينات للتشخيص
                if frame_count % 15 == 0:
                    angle_samples.append(main_angle)

                # استخدام العد التكيفي المحسن
                new_stage, rep_detected, last_rep_frame = count_reps_adaptive(
                    all_angles,
                    exercise_name,
                    stage,
                    last_rep_frame,
                    frame_count,
                    fps // 2,
                    is_lower_body
                )

                if rep_detected:
                    reps += 1
                    print(f"✅ تكرار {reps} (الزاوية: {main_angle:.1f}°)")

                stage = new_stage

                # معلومات تشخيصية
                debug = ""
                if len(all_angles) > 20:
                    recent = all_angles[-20:]
                    debug = f"Range: {max(recent):.0f}-{min(recent):.0f}"

                angle_type = category.replace('_based', '').title()
                frame = draw_info(frame, exercise_name, main_angle, reps, stage,
                                 ex_info['min_angle'], ex_info['max_angle'], angle_type, debug, all_angles)

        out.write(frame)
        frame_count += 1

        # عرض التقدم
        if frame_count % 30 == 0:
            progress = (frame_count / total_frames) * 100
            print(f"📊 التقدم: {progress:.1f}% - التكرارات: {reps}", end='\r')

    cap.release()
    out.release()
    pose.close()

    print(f"\n✅ اكتمل التحليل! إجمالي التكرارات: {reps}")

    # تحليل الزوايا للتشخيص
    if all_angles:
        min_ang = min(all_angles)
        max_ang = max(all_angles)
        avg_ang = np.mean(all_angles)
        std_ang = np.std(all_angles)

        # تحليل زوايا كل ركبة على حدة
        if left_knee_angles and right_knee_angles:
            left_min = min(left_knee_angles)
            left_max = max(left_knee_angles)
            right_min = min(right_knee_angles)
            right_max = max(right_knee_angles)

            print(f"\n📊 تحليل الركبتين:")
            print(f"   الركبة اليسرى: {left_min:.1f}° - {left_max:.1f}°")
            print(f"   الركبة اليمنى: {right_min:.1f}° - {right_max:.1f}°")

        # تشخيص المشكلة
        print(f"\n{'='*60}")
        print("🔍 تشخيص الحركة:")
        print(f"{'='*60}")
        print(f"📊 نطاق الزوايا الفعلي: {min_ang:.1f}° - {max_ang:.1f}°")
        print(f"📏 مدى الحركة: {max_ang - min_ang:.1f}°")
        print(f"🎯 العتبات المطلوبة: النزول < {ex_info['down_threshold']}°, الصعود > {ex_info['up_threshold']}°")

        if max_ang < ex_info['up_threshold']:
            print(f"⚠️ المشكلة: الزاوية العليا ({max_ang:.1f}°) أقل من العتبة ({ex_info['up_threshold']}°)")
            print(f"   الحل: اخفض العتبة إلى {max_ang - 5:.0f}°")

        if min_ang > ex_info['down_threshold']:
            print(f"⚠️ المشكلة: الزاوية السفلى ({min_ang:.1f}°) أعلى من العتبة ({ex_info['down_threshold']}°)")
            print(f"   الحل: ارفع العتبة إلى {min_ang + 5:.0f}°")

        # اقتراح عتبات جديدة
        suggested_up = max_ang - 5
        suggested_down = min_ang + 5
        print(f"\n💡 عتبات مقترحة للتمرين:")
        print(f"   up_threshold: {suggested_up:.0f}")
        print(f"   down_threshold: {suggested_down:.0f}")

    else:
        min_ang = max_ang = avg_ang = std_ang = 0

    # عرض النتائج النهائية
    print(f"\n{'='*60}")
    print("📊 النتائج النهائية:")
    print(f"{'='*60}")
    print(f"📌 التمرين: {exercise_name}")
    print(f"📂 الفئة: {category}")
    print(f"🔄 التكرارات: {reps}")
    print(f"📐 أقل زاوية: {min_ang:.1f}°")
    print(f"📐 أكبر زاوية: {max_ang:.1f}°")
    print(f"📊 متوسط الزاوية: {avg_ang:.1f}°")
    print(f"📈 انحراف معياري: {std_ang:.1f}°")
    print(f"📏 مدى الحركة: {max_ang - min_ang:.1f}°")

    # توليد تقييم الذكاء الاصطناعي
    if reps > 0:
        print(f"\n{'='*60}")
        print("🧠 جاري توليد تقييم الذكاء الاصطناعي...")
        print(f"{'='*60}")

        ai_feedback = generate_ai_feedback(
            exercise_name, category,
            ex_info['min_angle'], ex_info['max_angle'],
            min_ang, max_ang,
            reps
        )

        print(f"\n{ai_feedback}")
        print(f"\n{'='*60}")

        # حفظ التقرير
        report_path = f"/content/{exercise_name}_ai_report.txt"
        with open(report_path, 'w', encoding='utf-8') as f:
            f.write("="*60 + "\n")
            f.write(f"تقرير تمرين: {exercise_name}\n")
            f.write("="*60 + "\n\n")
            f.write(f"التمرين: {exercise_name}\n")
            f.write(f"الفئة: {category}\n")
            f.write(f"الوصف: {ex_info['description']}\n")
            f.write(f"التكرارات: {reps}\n")
            f.write(f"أقل زاوية: {min_ang:.1f}°\n")
            f.write(f"أكبر زاوية: {max_ang:.1f}°\n")
            f.write(f"متوسط الزاوية: {avg_ang:.1f}°\n")
            f.write(f"مدى الحركة: {max_ang - min_ang:.1f}°\n\n")
            f.write("="*60 + "\n")
            f.write("تقييم الذكاء الاصطناعي:\n")
            f.write("="*60 + "\n")
            f.write(ai_feedback)

        print(f"📁 تم حفظ التقرير في: {report_path}")

        # إنشاء ملف صوتي
        print(f"\n{'='*60}")
        print("🎤 إنشاء الملف الصوتي...")
        print(f"{'='*60}")

        spoken_text = extract_feedback_for_tts(ai_feedback)
        audio_file = f"/content/{exercise_name}_feedback.mp3"

        try:
            import nest_asyncio
            nest_asyncio.apply()
            loop = asyncio.get_event_loop()
            loop.run_until_complete(create_coach_audio(spoken_text, audio_file))

            # دمج الصوت مع الفيديو
            print(f"\n{'='*60}")
            print("🎬 إنشاء الفيديو النهائي بالصوت...")
            print(f"{'='*60}")

            final_video = f"/content/{exercise_name}_final.mp4"
            result = add_audio_to_video(temp_video, audio_file, final_video)

            if result:
                final_path = result
                print(f"✅ تم إنشاء الفيديو النهائي: {final_path}")
            else:
                final_path = temp_video
                print(f"⚠️ تم استخدام الفيديو بدون صوت: {final_path}")

        except Exception as e:
            print(f"⚠️ لم يتم إنشاء الصوت: {e}")
            final_path = temp_video
    else:
        final_path = temp_video
        ai_feedback = "لم يتم اكتشاف تكرارات"
        report_path = None

    return reps, final_path, report_path, ai_feedback

# ============================================================================
# التشغيل - أدخل بياناتك هنا
# ============================================================================

# ✅ اسم التمرين
EXERCISE_NAME = "lunge"  # اختر من القائمة أعلاه

# ✅ مسار الفيديو
VIDEO_PATH = '/content/drive/MyDrive/graduation_project/Lunges_test.mp4'

# تشغيل التحليل مع تقييم الذكاء الاصطناعي
reps, video_path, report_path, ai_feedback = analyze_with_ai_feedback(VIDEO_PATH, EXERCISE_NAME)

# عرض الفيديو النهائي
if os.path.exists(video_path):
    print(f"\n🎬 تشغيل الفيديو الناتج:")
    display(Video(video_path, width=640))

# تشغيل الصوت
audio_file = f"/content/{EXERCISE_NAME}_feedback.mp3"
if os.path.exists(audio_file):
    print(f"\n🔊 تشغيل التعليق الصوتي:")
    display(Audio(audio_file, autoplay=True))

ImportError: cannot import name 'runtime_version' from 'google.protobuf' (/usr/local/lib/python3.12/dist-packages/google/protobuf/__init__.py)

In [ ]:
import os
import json
from google.colab import drive
from safetensors.torch import load_file

# 1. ربط قوقل درايف بالـ Colab
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:

# 2. المسار المباشر للفولدر بتاعك بناءً على الصورة (Shared with me بيقرأ كـ .shortcut-targets-by-id أو MyDrive تلقائي في كولاب)
# يفضل تروح في القائمة الجانبية بالـ Colab وتتأكد من المسار، لكن ده المسار القياسي:
adapter_dir = '/content/drive/MyDrive/graduation_project/rehab_coach_lora_FINAL'

# لو الفولدر مظهرش بالمسار اللي فوق بسبب أنه Shared مع درايفك، جرب المسار البديل ده:
# adapter_dir = '/content/drive/Shareddrives/graduation_project/rehab_coach_lora_FINAL'

print("-" * 60)
# التأكد من قراءة الفولدر والملفات
if os.path.exists(adapter_dir):
    print("✅ تم الوصول للمجلد بنجاح! الملفات المتاحة هي:")
    print(os.listdir(adapter_dir))
else:
    print("❌ المسار غير صحيح. يرجى فتح قائمة الملفات في كولاب (أيقونة الفولدر على اليسار) ونسخ المسار الصحيح (Copy path).")
print("-" * 60)


# 3. قراءة ملف الـ adapter_config.json
try:
    with open(f"{adapter_dir}/adapter_config.json", "r", encoding="utf-8") as f:
        config_data = json.load(f)
    print("\n📄 [1] محتوى ملف adapter_config.json:")
    print(json.dumps(config_data, indent=4))
except Exception as e:
    print(f"خيار قراءة الـ config فشل: {e}")


# 4. قراءة ملف الـ tokenizer_config.json
try:
    with open(f"{adapter_dir}/tokenizer_config.json", "r", encoding="utf-8") as f:
        tok_config = json.load(f)
    print("\n📄 [2] محتوى ملف tokenizer_config.json (مقتطف):")
    # طباعة أول 10 عناصر فقط علشان حجم الملف ميكونش ضخم في الـ output
    print(json.dumps(dict(list(tok_config.items())[:10]), indent=4))
except Exception as e:
    print(f"خيار قراءة الـ tokenizer config فشل: {e}")


# 5. فحص أوزان ملف الـ adapter_model.safetensors (طباعة أسماء الطبقات/المصفوفات فقط)
try:
    print("\n📊 [3] فحص ملف الأوزان adapter_model.safetensors:")
    weights = load_file(f"{adapter_dir}/adapter_model.safetensors")
    print(f"عدد المصفوفات (Tensors) داخل الملف: {len(weights.keys())}")
    print("أول 5 طبقات (Layers) في الـ LoRA:")
    for key in list(weights.keys())[:5]:
        print(f" - {key}: {weights[key].shape}")
except Exception as e:
    print(f"خيار فحص الـ safetensors فشل: {e}")


# 6. قراءة ملف الـ README.md (لو حابب تشوف كاتبين فيه إيه عن المشروع)
try:
    if os.path.exists(f"{adapter_dir}/README.md"):
        print("\n📝 [4] محتوى ملف README.md:")
        with open(f"{adapter_dir}/README.md", "r", encoding="utf-8") as f:
            print(f.read()[:500]) # هيطبع أول 500 حرف
except Exception as e:
    print(f"خيار قراءة الـ README فشل: {e}")

------------------------------------------------------------
✅ تم الوصول للمجلد بنجاح! الملفات المتاحة هي:
['chat_template.jinja', 'adapter_model.safetensors', 'adapter_config.json', 'tokenizer_config.json', 'README.md', 'tokenizer.json', 'نسخة من tokenizer_config.json', 'نسخة من adapter_config.json', 'نسخة من tokenizer.json', 'نسخة من README.md', 'نسخة من adapter_model.safetensors', 'نسخة من chat_template.jinja']
------------------------------------------------------------

📄 [1] محتوى ملف adapter_config.json:
{
    "alora_invocation_tokens": null,
    "alpha_pattern": {},
    "arrow_config": null,
    "auto_mapping": {
        "base_model_class": "Qwen2ForCausalLM",
        "parent_library": "transformers.models.qwen2.modeling_qwen2",
        "unsloth_fixed": true
    },
    "base_model_name_or_path": "unsloth/qwen2.5-7b-instruct-bnb-4bit",
    "bias": "none",
    "corda_config": null,
    "ensure_weight_tying": false,
    "eva_config": null,
    "exclude_modules": null,
    "fan_in_

```markdown
### Model File and Kernel State Verification Summary

**1. Model Files Presence:**
The output from `!ls -F /content/drive/MyDrive/graduation_project/rehab_coach_lora_FINAL` clearly shows the presence of essential model configuration and weight files:
* `adapter_config.json@`
* `adapter_model.safetensors@`
* `tokenizer_config.json@`
* `tokenizer.json@`

These files are indicative of a complete LoRA (fine-tuned) model structure, meaning the necessary files are indeed in the specified directory.

**2. Kernel State for `model` and `tokenizer` objects:**
Although `model_path` is present in the kernel state as a string variable, previous execution attempts (as indicated by the `RuntimeError` in cells `gllfnNc96st4` and `QaKFdfTq-cCs`) show that the `FastLanguageModel.from_pretrained` function failed to successfully load the model into `model` and `tokenizer` *objects*. The error `RuntimeError: Unsloth: No config file found - are you sure the 'model_name' is correct?` suggests that while the files exist, there might be an issue with how `unsloth` interprets the directory or its contents during the loading process, preventing the model from being instantiated as a Python object in the kernel.

Therefore, while the files are physically present, the `model` and `tokenizer` *objects* are not successfully loaded in the kernel at this point, which is crucial for model inference.
```

## Final Task

### Subtask:
Report whether the `rehab_coach_lora_FINAL` folder appears to contain the necessary model files for `FastLanguageModel.from_pretrained`.


## Summary:

### Q&A
Yes, the `rehab_coach_lora_FINAL` folder appears to contain the necessary model files, including `adapter_config.json@`, `adapter_model.safetensors@`, `tokenizer_config.json@`, and `tokenizer.json@`, which are indicative of a complete LoRA (fine-tuned) model structure. However, despite the physical presence of these files, previous attempts to load them using `FastLanguageModel.from_pretrained` resulted in an error, indicating the model and tokenizer objects were not successfully instantiated in the kernel.

### Data Analysis Key Findings
*   The `/content/drive/MyDrive/graduation_project/rehab_coach_lora_FINAL` directory contains essential model configuration and weight files: `adapter_config.json@`, `adapter_model.safetensors@`, `tokenizer_config.json@`, and `tokenizer.json@`, along with `chat_template.jinja@` and `README.md`. The `@` suffix indicates some are symbolic links.
*   These identified files suggest the presence of a complete LoRA fine-tuned model structure.
*   Although the necessary model files are physically present in the directory, the `model` and `tokenizer` objects were not successfully loaded into the kernel. Attempts to load them with `FastLanguageModel.from_pretrained` resulted in a `RuntimeError: Unsloth: No config file found - are you sure the 'model_name' is correct?`.

### Insights or Next Steps
*   Investigate the `RuntimeError` to understand why `FastLanguageModel.from_pretrained` fails to find a config file despite `adapter_config.json` being present. This may involve checking expected filenames (`config.json` vs. `adapter_config.json`) or resolving issues related to symbolic links.
*   Verify the correct usage and parameters for `FastLanguageModel.from_pretrained` when loading a LoRA adapter from a local directory, ensuring compatibility with the existing file structure.
